In [ ]:
import os
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import optuna
from datetime import datetime

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight

# --- KONFIGURASI FOLDER KHUSUS TUNING ---
DATASET_DIR = pathlib.Path("data_cropped") # Tetap membaca dari sumber yang sama
TUNING_DIR = "HASIL_TUNING_OPTUNA"
LOG_FILE = os.path.join(TUNING_DIR, "log_eksperimen_tuning.csv")
IMG_SIZE = 224

# Buat folder khusus tuning jika belum ada
os.makedirs(TUNING_DIR, exist_ok=True)

# Pemuatan Path Data
all_image_paths = [str(p) for p in DATASET_DIR.glob('*/*.png')]
class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
all_image_labels = [label_to_index[pathlib.Path(p).parent.name] for p in all_image_paths]

X = np.array(all_image_paths)
y = np.array(all_image_labels)

# --- FUNGSI PIPELINE DATA ---
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=4, expand_animations=False)
    img = tf.cast(img, tf.float32)
    rgb = img[..., :3]
    alpha = img[..., 3:4] / 255.0
    img_fixed = rgb * alpha
    img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
    img_final = img_resized / 255.0
    img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img_final, tf.one_hot(label, depth=len(class_names_list))

# --- FOCAL LOSS KUSTOM ---
def categorical_focal_loss(alpha, gamma):
    alpha_tensor = tf.constant(alpha, dtype=tf.float32)
    def focal_loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha_tensor * tf.math.pow(1.0 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=-1))
    return focal_loss

# --- FUNGSI BUILD MODEL ---
def build_model(dense_units, dropout_rate):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    max_p = layers.GlobalMaxPooling2D()(x)
    avg_p = layers.GlobalAveragePooling2D()(x)
    merged = layers.Concatenate()([max_p, avg_p])
    
    x = layers.Dense(dense_units, activation='relu')(merged)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(4, activation='softmax')(x)
    
    return models.Model(inputs=inputs, outputs=outputs)

# --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
# --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
def objective(trial):
    # 1. Optuna Memilih Angka Kombinasi
    lr = trial.suggest_categorical("learning_rate", [5e-5, 1e-4, 5e-4, 1e-3])
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    dense_units = trial.suggest_categorical("dense_units", [128, 256])
    dropout_rate = trial.suggest_categorical("dropout_rate", [0.2, 0.3, 0.4])
    gamma_val = trial.suggest_categorical("gamma", [0.0, 1.0, 2.0]) 

    # --- TAMBAHAN PRINT INFO TRIAL ---
    print(f"\n" + "-"*50)
    print(f"▶️ MEMULAI TRIAL KE-{trial.number}")
    print(f"⚙️ Parameter: LR={lr}, Batch={batch_size}, Dense={dense_units}, Dropout={dropout_rate}, Gamma={gamma_val}")
    print("-"*50)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []

    # 2. Uji Kombinasi ke dalam 5 Lipatan (K-Fold)
    for fold_no, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        alpha_list = [float(w) for w in weights]

        train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).map(process_path).shuffle(len(X_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
        val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(process_path).batch(batch_size).prefetch(tf.data.AUTOTUNE)

        tf.keras.backend.clear_session()
        model = build_model(dense_units, dropout_rate)
        
        loss_fn = categorical_focal_loss(alpha=alpha_list, gamma=gamma_val)
        model.compile(optimizer=Adam(learning_rate=lr), loss=loss_fn, metrics=['accuracy'])

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True), 
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6) 
        ]

        model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=0)
        
        y_val_true = np.concatenate([lbl for img, lbl in val_ds], axis=0)
        y_val_pred = model.predict(val_ds, verbose=0)
        acc = accuracy_score(np.argmax(y_val_true, axis=1), np.argmax(y_val_pred, axis=1))
        fold_accs.append(acc)
        
        # --- TAMBAHAN PRINT INFO PER FOLD ---
        print(f"   ✓ Fold {fold_no} Selesai | Akurasi: {acc*100:.2f}%")

    # 3. Dapatkan Rata-rata Akurasi dari 5 Lipatan
    avg_accuracy = np.mean(fold_accs)
    
    # --- TAMBAHAN PRINT KESIMPULAN TRIAL ---
    print(f"🏁 KESIMPULAN TRIAL {trial.number} | Rata-rata Akurasi: {avg_accuracy*100:.2f}%")
    
    # --- PENCATATAN LOG KE CSV ---
    log_data = {
        "Trial_No": trial.number,
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Learning_Rate": lr,
        "Batch_Size": batch_size,
        "Dense_Units": dense_units,
        "Dropout_Rate": dropout_rate,
        "Gamma": gamma_val,
        "Avg_Accuracy": round(avg_accuracy * 100, 2)
    }
    df_log = pd.DataFrame([log_data])
    if not os.path.exists(LOG_FILE):
        df_log.to_csv(LOG_FILE, index=False)
    else:
        df_log.to_csv(LOG_FILE, mode='a', header=False, index=False)

    return avg_accuracy

# --- EKSEKUSI TUNING ---
print("\n[INFO] Memulai Proses Hyperparameter Tuning...")
print(f"[INFO] Hasil akan dicatat secara real-time di: {LOG_FILE}\n")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("\n" + "="*60)
print("🏆 HASIL TUNING TERBAIK DITEMUKAN!")
print(f"  Akurasi Rata-rata Tertinggi: {study.best_value*100:.2f}%")
print(f"  Coba Parameter Ini di Script Final Anda:")
for key, value in study.best_params.items():
    print(f"    - {key}: {value}")
print("="*60)

[I 2026-04-13 09:46:50,751] A new study created in memory with name: no-name-56fe5b42-b3d4-4b7d-8d2e-e26f93a79165


[I 2026-04-13 11:04:33,843] Trial 0 finished with value: 0.5596992019643954 and parameters: {'learning_rate': 0.001, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.4, 'gamma': 2.0}. Best is trial 0 with value: 0.5596992019643954.
[W 2026-04-13 12:00:44,008] Trial 1 failed with parameters: {'learning_rate': 5e-05, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.3, 'gamma': 0.0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\PDE\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\PDE\AppData\Local\Temp\ipykernel_32936\3905134475.py", line 124, in objective
    model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=0)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PDE\AppData\Local\Programs\Python\Python313\Lib\site-pac

KeyboardInterrupt: 